In [1]:
import pandas as pd
import numpy as np  
import warnings
import re

warnings.filterwarnings('ignore')

raw_df = pd.read_excel('raw/푸드테크_H-PEACE_식생활관리_0923_최수연 - 전달.xlsx').drop(0, axis=0)
var_df = pd.read_excel('변수 정의.xlsx')

In [2]:
raw_df['수진일'].max()

'2023-05-31'

In [3]:
info_var = ['R-ID', '수진일', '성별', '나이', '신장']
plus_var = ['최종학력', '결혼상태', '가계수입']
health_var = ['SBP', 'DBP', 'CHOL.', 'TG', 'LDL CHOL.', 'HDL CHOL.', 'GLUCOSE', 'HBA1C', 'eGFR', '허리둘레(WAIST)', '체중', '체질량지수']
disease_var = var_df[(var_df['분류'] == '질환')]['컬럼명1'].tolist()
medication_var = ['고혈압_투약여부','당뇨_투약여부', '고지혈증_투약여부']
smoke_var = var_df[(var_df['분류'] == '흡연')]['컬럼명1'].tolist()

diet_var = var_df[var_df['분류']=='식습관']
diet_var1 = diet_var['컬럼명1'].tolist()
diet_var2 = diet_var['컬럼명2'].tolist()

activity_var = var_df[var_df['분류']=='운동']['컬럼명1'].tolist()
drinking_var = var_df[(var_df['분류'] == '음주')]['컬럼명1'].tolist()

## 식습관 변수 통합

In [4]:
diet_df = raw_df[diet_var1 + diet_var2 + ['R-ID', '수진일']]

diet_var_g1 = diet_var.groupby('소분류')['컬럼명1'].apply(list).to_dict()
diet_var_g2 = diet_var.groupby('소분류')['컬럼명2'].apply(list).to_dict()

data_df1 = diet_df[diet_var1 + ['R-ID', '수진일']]
data_df2 = diet_df[diet_var2 + ['R-ID', '수진일']]

invalid_ids = set()

for subcat, cols in diet_var_g1.items():
    valid_cols = [c for c in cols if c in data_df1.columns]
    if not valid_cols:
        continue
    dup_mask = data_df1[valid_cols].notna().sum(axis=1) > 1
    dup_ids = set(data_df1.loc[dup_mask, 'R-ID'])
    invalid_ids.update(dup_ids)

for subcat, cols in diet_var_g2.items():
    valid_cols = [c for c in cols if c in data_df2.columns]
    if not valid_cols:
        continue
    dup_mask = data_df2[valid_cols].notna().sum(axis=1) > 1
    dup_ids = set(data_df2.loc[dup_mask, 'R-ID'])
    invalid_ids.update(dup_ids)

data_df1 = data_df1[~data_df1['R-ID'].isin(invalid_ids)]
data_df2 = data_df2[~data_df2['R-ID'].isin(invalid_ids)]

# score 맵핑
score_map1 = diet_var.set_index('컬럼명1')['score'].dropna()
score_map2 = diet_var.set_index('컬럼명2')['score'].dropna()

# diet_df1: 컬럼명1 기준
diet_df1 = pd.DataFrame(index=data_df1.index)
for subcategory, columns in diet_var_g1.items():
    valid_columns = [col for col in columns if col in data_df1.columns]
    diet_df1[subcategory] = (
        data_df1[valid_columns]
        .apply(lambda row: next(
            (score_map1.get(col) for col in valid_columns if pd.notna(row[col])), 
            np.nan
        ), axis=1)
    )

# diet_df2: 컬럼명2 기준
diet_df2 = pd.DataFrame(index=data_df2.index)
for subcategory, columns in diet_var_g2.items():
    valid_columns = [col for col in columns if col in data_df2.columns]
    diet_df2[subcategory] = (
        data_df2[valid_columns]
        .apply(lambda row: next(
            (score_map2.get(col) for col in valid_columns if pd.notna(row[col])), 
            np.nan
        ), axis=1)
    )

# 결합 및 최종 결과 생성
diet_df_cleaned = diet_df[~diet_df['R-ID'].isin(invalid_ids)]
diet_df_filled = pd.concat([diet_df2.fillna(diet_df1), diet_df_cleaned[['R-ID', '수진일']]],axis=1).dropna()

## 질환 변수 통합

In [5]:
info_df = raw_df[info_var]
disease_df_filled = raw_df[['R-ID', '수진일'] + health_var + disease_var + medication_var].dropna()
diet_disease_df_filled = disease_df_filled.merge(diet_df_filled, on=['R-ID', '수진일'], how='inner').merge(info_df, on=['R-ID', '수진일'], how='inner')

# 'nonHDLC' 추가
diet_disease_df_filled['CHOL.'] = pd.to_numeric(diet_disease_df_filled['CHOL.'], errors='coerce')
diet_disease_df_filled['HDL CHOL.'] = pd.to_numeric(diet_disease_df_filled['HDL CHOL.'], errors='coerce')
diet_disease_df_filled['nonHDLC'] = diet_disease_df_filled['CHOL.'] - diet_disease_df_filled['HDL CHOL.']

# 비만 추가
diet_disease_df_filled['비만'] = np.where(
    diet_disease_df_filled['체질량지수'] < 23, 0,
    np.where(diet_disease_df_filled['체질량지수'] >= 25, 2, 1)
)

# BMI category 추가
diet_disease_df_filled['BMI category'] = np.where(
    diet_disease_df_filled['체질량지수'] < 18.5, 0,
        np.where(diet_disease_df_filled['체질량지수'] < 23, 1,
            np.where(diet_disease_df_filled['체질량지수'] < 25, 2,
                np.where(diet_disease_df_filled['체질량지수'] < 30, 3, 4)
            )
        )
    )

# WC 추가
diet_disease_df_filled['허리둘레(WAIST)'] = pd.to_numeric(diet_disease_df_filled['허리둘레(WAIST)'], errors='coerce')
diet_disease_df_filled['WC (M>=90, F>=85)'] = np.where(
    ((diet_disease_df_filled['허리둘레(WAIST)'] >= 90) & (diet_disease_df_filled['성별'] == 'M')) |
    ((diet_disease_df_filled['허리둘레(WAIST)'] >= 85) & (diet_disease_df_filled['성별'] == 'F')), 1, 0
)

# 만성콩팥병 추가
diet_disease_df_filled['Chronic kidney disease (eGFR<60)'] = np.where(
    diet_disease_df_filled['eGFR'] < 60, 1,
    np.where(diet_disease_df_filled['eGFR'] >= 60, 0, np.nan)
).astype(int)

In [6]:
len(diet_disease_df_filled)

60504

## MetS

In [7]:
mets_cols = ['허리둘레(WAIST)', 'SBP', 'DBP', '고혈압_투약여부', 'GLUCOSE', '당뇨_투약여부', 'TG', '고지혈증_투약여부', 'HDL CHOL.']
MetS_df = raw_df[mets_cols + ['R-ID', '성별', '수진일']]
for col in mets_cols:
    MetS_df[col] = MetS_df[col].apply(pd.to_numeric, errors='coerce')

In [8]:
# 성별에 따른 허리둘레 기준 설정
MetS_df['Increased waist circumference'] = (
    ((MetS_df['성별'] == 'M') & (MetS_df['허리둘레(WAIST)'].astype(float) >= 90)) | 
    ((MetS_df['성별'] == 'F') & (MetS_df['허리둘레(WAIST)'].astype(float) >= 85))
)

# 혈압 기준 설정
MetS_df['Elevated blood pressure'] = (
    ((MetS_df['SBP'].astype(float) >= 130) | 
     (MetS_df['DBP'].astype(float) >= 85)) | 
    (MetS_df['고혈압_투약여부'] == 1)
)

# 공복혈당 기준 설정
MetS_df['Impaired fasting glucose'] = (
    (MetS_df['GLUCOSE'].astype(float) >= 100) | 
    (MetS_df['당뇨_투약여부'] == 1)
)

# 중성지방 기준 설정
MetS_df['Elevated triglycerides'] = (
    (MetS_df['TG'].astype(float) >= 150) | 
    (MetS_df['고지혈증_투약여부'] == 1)
)

# 성별에 따른 HDL 콜레스테롤 기준 설정
MetS_df['Decreased HDL-C'] = (
    ((MetS_df['성별'] == 'M') & (MetS_df['HDL CHOL.'].astype(float) < 40)) | 
    ((MetS_df['성별'] == 'F') & (MetS_df['HDL CHOL.'].astype(float) < 50))
)

mets_tf = ['Increased waist circumference', 'Elevated blood pressure', 'Impaired fasting glucose', 'Elevated triglycerides', 'Decreased HDL-C']
MetS_df['MetS'] = (MetS_df[mets_tf] == 1).sum(axis=1).apply(lambda x: 1 if x >= 3 else 0)

In [9]:
sum(MetS_df['MetS'])

24469

## 운동 변수 통합

In [10]:
activity_var1 = activity_var[:6]
activity_var2 = activity_var[9:]

activity_df1 = raw_df[activity_var1+ ['R-ID', '수진일']]
activity_df2 = raw_df[activity_var2+ ['R-ID', '수진일']]

activity_df_filled = activity_df2.fillna(activity_df1)

for col in activity_var2:
    activity_df_filled[col] = pd.to_numeric(activity_df_filled[col], errors="coerce")

activity_df_filled["중강도_총시간"] = activity_df_filled["중강도_운동빈도 [19ver only]"] * activity_df_filled["중강도_운동시간 [19ver only]"]
activity_df_filled["고강도_총시간"] = activity_df_filled["고강도_운동빈도 [19ver only]"] * activity_df_filled["고강도_운동시간 [19ver only]"]
activity_df_filled["총_운동시간"] = activity_df_filled["중강도_총시간"] + activity_df_filled["고강도_총시간"]

def categorize_activity(row):
    if row["중강도_운동여부 [19ver only]"] == 0 and row["고강도_운동여부 [19ver only]"] == 0:
        return 0
    elif (1 <= row["중강도_총시간"] < 150 or 1 <= row["고강도_총시간"] < 75 or 1 <= row["총_운동시간"] < 150):
        return 1
    elif (row["중강도_총시간"] >= 150 or row["고강도_총시간"] >= 75 or row["총_운동시간"] >= 150):
        return 2
    return np.nan

activity_df_filled["활동량"] = activity_df_filled.apply(categorize_activity, axis=1)
activity_df_filled = activity_df_filled[['활동량', 'R-ID', '수진일']].dropna()#.fillna('Missing value')

## 음주 변수 통합

In [11]:
drinking_df = raw_df[drinking_var+ ['R-ID', '수진일']]

# 음주 정의
ALCOHOL_PER_GLASS = 8  # 1잔 = 8g 알코올
FREQ_MAPPING = {
    0: 0,  # 월 1회 이하 (0회/주 로 계산)
    1: 0.75,  # 월 2-4회 (평균 0.75회/주로 계산)
    2: 2,     # 주 2회
    3: 3,     # 주 3회
    4: 4,     # 주 4회
    5: 5,     # 주 5회
    6: 6,     # 주 6회
    7: 7      # 주 7회
}
DRINK_MAPPING = {
    0: 2,    # 2잔
    1: 4,    # 4잔
    2: 6,    # 6잔
    3: 9,    # 9잔
    4: 10    # 10잔 이상
}

# 음주량 계산 및 카테고리 분류 함수
def classify_alcohol_intake(row):
    if pd.isna(row['음주빈도']) or pd.isna(row['음주량']):
        return np.nan
    
    weekly_frequency = FREQ_MAPPING.get(row['음주빈도'], np.nan)
    drinks_per_session = DRINK_MAPPING.get(row['음주량'], np.nan)
    
    if pd.isna(weekly_frequency) or pd.isna(drinks_per_session):
        return np.nan
    
    weekly_alcohol_intake = weekly_frequency * drinks_per_session * ALCOHOL_PER_GLASS
    
    if weekly_alcohol_intake == 0:
        return 0
    elif weekly_alcohol_intake < 210:
        return 1
    else:
        return 2

drinking_df['음주빈도'] = pd.to_numeric(drinking_df['음주빈도'], errors='coerce')
drinking_df['음주량'] = pd.to_numeric(drinking_df['음주량'], errors='coerce')

drinking_df['음주'] = drinking_df.apply(classify_alcohol_intake, axis=1)
drinking_df = drinking_df[['음주', 'R-ID', '수진일']].dropna()#.fillna('Missing value')

## 흡연 변수 통합

In [12]:
smoke_df = raw_df[smoke_var + ['R-ID', '수진일']]

# 0=비흡연, 1=과거흡연, 2=현재흡연, na -> 0=현재 비흡연, 1=현재흡연, na=na
for col in smoke_var:
    smoke_df[col] = smoke_df[col].apply(
        lambda x: 1 if x == 2 else (0 if pd.notna(x) else np.nan)
    )

smoke_df = smoke_df.dropna()

## 전체 통합

In [13]:
plus_df = raw_df[plus_var + ['R-ID', '수진일']].dropna()#.fillna('Missing value')

In [14]:
# 시작점
print(f"diet_df_filled: {len(diet_df_filled)}")
print(f"disease_df_filled: {len(disease_df_filled)}")
print(f"activity_df_filled: {len(activity_df_filled)}")
print(f"drinking_df: {len(drinking_df)}")
print(f"smoke_df: {len(smoke_df)}")
print(f"plus_df: {len(plus_df)}")
print(f"MetS_df: {len(MetS_df)}")

# Step 1: diet+disease merge
print(f"diet_df_filled: {len(diet_df_filled)}")

step1 = diet_disease_df_filled
print(f"After disease merge: {len(step1)}")

# Step 2: activity merge  
step2 = step1.merge(activity_df_filled, on=['R-ID', '수진일'], how='inner')
print(f"After activity merge: {len(step2)}")

# Step 3: drinking merge
step3 = step2.merge(drinking_df, on=['R-ID', '수진일'], how='inner')
print(f"After drinking merge: {len(step3)}")

# Step 4: smoke merge
step4 = step3.merge(smoke_df, on=['R-ID', '수진일'], how='inner')
print(f"After smoke merge: {len(step4)}")

# Step 5: plus merge
step5 = step4.merge(plus_df, on=['R-ID', '수진일'], how='inner')
print(f"After socioeconomic merge: {len(step5)}")

# Step 6: MetS merge
total_df = step5.merge(MetS_df, on=['R-ID', '수진일'], how='inner', suffixes=('', '_y')).filter(regex='^(?!.*_y$)')
print(f"After MetS merge (final): {len(total_df)}")

# 중복 제거 후
total_df.sort_values(by='수진일', ascending=True, inplace=True)
total_df_only = total_df.drop_duplicates(subset='R-ID', keep='first').set_index('R-ID')
print(f"After removing duplicates: {len(total_df_only)}")

# 각 단계별 대상자 수 확인을 위한 코드
print(f"Original data size: {len(raw_df)}")
print(f"Final merged data: {len(total_df)}")
print(f"After removing duplicates: {len(total_df_only)}")

diet_df_filled: 78027
disease_df_filled: 68417
activity_df_filled: 48807
drinking_df: 80055
smoke_df: 85714
plus_df: 68208
MetS_df: 111985
diet_df_filled: 78027
After disease merge: 60504
After activity merge: 37319
After drinking merge: 37277
After smoke merge: 37226
After socioeconomic merge: 27910
After MetS merge (final): 27910
After removing duplicates: 23040
Original data size: 111985
Final merged data: 27910
After removing duplicates: 23040


In [15]:
korean_to_paper = {
    # Basic information
    'R-ID': 'R-ID',
    '나이': 'Age',

    '성별': 'Sex',
    '최종학력': 'Education Level',
    '결혼상태': 'Marital Status', 
    '가계수입': 'Household Income',
    
    'BMI category' : 'BMI category', 
    'WC (M>=90, F>=85)' : 'WC (M>=90, F>=85)',
    
    # Anthropometric measures
    '신장': 'Height (cm)',
    '체중': 'Weight (kg)',
    '체질량지수': 'BMI(kg/m2)',
    '허리둘레(WAIST)': 'Waist circumference (cm)',
        
    # Disease status and medication
    '고혈압_통합': 'Hypertension',
    '고혈압_투약여부': 'Antihypertensive Medication',
    '당뇨_통합': 'Diabetes Mellitus',
    '당뇨_투약여부': 'Antidiabetic Medication',
    '고지혈증_통합': 'Hyperlipidemia',
    '고지혈증_투약여부': 'Statin Medication',
    '협심증/심근경색증_통합': 'Angina/Myocardial Infarction',
    '뇌졸중(중풍)_통합': 'Stroke',
    'Chronic kidney disease (eGFR<60)': 'Chronic kidney disease (eGFR<60)',

    # Clinical measures
    'SBP': 'Systolic blood pressure (mmHg)',
    'DBP': 'Diastolic Blood Pressure (mmHg)',
    'CHOL.' : 'Total Cholesterol (mg/dL)',
    'TG': 'Triglycerides (mg/dL)',
    'LDL CHOL.': 'LDL-C (mg/dL)',
    'HDL CHOL.': 'HDL-C (mg/dL)',
    'nonHDLC': 'Non-HDL-C (mg/dL)',
    'GLUCOSE': 'Fasting glucose (mg/dL)',
    'HBA1C': 'HbA1c (%)',
    'eGFR': 'eGFR (mL/min/1.73m2)',

    # Lifestyle risk component
    '비만' : 'Obese', # Obese (BMI ≥ 25 kg/m2), Overweight (BMI 23-24.9 kg/m2), Normal (BMI < 23 kg/m2)
    '활동량': 'Physical activity', # Inactive (0), Minimally active (1), Active (2)
    '음주': 'Alcohol Consumption', # Non-drinker (0), Moderate drinker (1), Heavy drinker (2)
    '일반담배_흡연여부': 'Current smoking', # Non-smoker (0), Current smoker (1)

    # Dietary patterns
    '식사 빈도': 'Meal Frequency',
    '식사량': 'Meal Portion Size',
    '외식빈도': 'Eating Out Frequency',
    '밥 양': 'Rice Portion Size',
    '간식빈도': 'Snacking Frequency',
    
    # Food groups
    '곡류': 'Grain Products',
    '단백질류': 'Protein Foods',
    '채소': 'Vegetables',
    '유제품': 'Dairy Products',
    '과일': 'Fruits',
    '튀김': 'Fried Foods',
    '고지방 육류': 'High Fat Meat',
    '인스턴트 가공식품': 'Processed Foods',
    
    # Beverages
    '물': 'Water Intake',
    '커피': 'Coffee Consumption',
    '음료류': 'Sugar-Sweetened Beverages',
    
    # Taste preferences
    '짠 간': 'Additional Salt Use',
    '짠 식습관': 'Salty Food Consumption',
    '단맛': 'Sweet Food Consumption',

    # MetS
    'Increased waist circumference' : 'Increased waist circumference', 
    'Elevated blood pressure' : 'Elevated blood pressure', 
    'Impaired fasting glucose' : 'Impaired fasting glucose', 
    'Elevated triglycerides' : 'Elevated triglycerides', 
    'Decreased HDL-C' : 'Decreased HDL-C',
    'MetS': 'MetS'
}

total_df_only.rename(columns=korean_to_paper, inplace=True)
total_df_only = total_df_only[list(korean_to_paper.values())[1:]]

total_df.rename(columns=korean_to_paper, inplace=True)
total_df = total_df[list(korean_to_paper.values())[1:]]
total_df_only.to_csv('processed_data/total_only_org.csv', index='R-ID')

In [16]:
def transform_values(df):
    # Grain Products: 3→5, 2→3, 1→1
    df['Grain Products'] = df['Grain Products'].replace({3: 5, 2: 3, 1: 1})
    
    # Protein Foods: 4→5, 3→3, 2 or 1→1
    df['Protein Foods'] = df['Protein Foods'].replace({4: 5, 2: 1, 1: 1})
    
    # Vegetables: 4→5, 3→3, 2 or 1→1
    df['Vegetables'] = df['Vegetables'].replace({4: 5, 2: 1, 1: 1})
    
    # Fruits: 3→5, 2→3, 1→1
    df['Fruits'] = df['Fruits'].replace({3: 5, 2: 3})
    
    # Dairy Products: 3→5, 4 or 2→3, 1→1
    df['Dairy Products'] = df['Dairy Products'].replace({3: 5, 4: 3, 2: 3})
    
    # Sweet Food Consumption: 1→5, 2→3, 3→1
    df['Sweet Food Consumption'] = df['Sweet Food Consumption'].replace({1: 5, 2: 3, 3: 1})
    
    # Fried Foods: 1→5, 2→3, 3 or 4→1
    df['Fried Foods'] = df['Fried Foods'].replace({1: 5, 2: 3, 3: 1, 4: 1})
    
    # High Fat Meat: 1→5, 2→3, 3 or 4→1
    df['High Fat Meat'] = df['High Fat Meat'].replace({1: 5, 2: 3, 3: 1, 4: 1})
    
    # Processed Foods: 1→5, 2→3, 3 or 4→1
    df['Processed Foods'] = df['Processed Foods'].replace({1: 5, 2: 3, 3: 1, 4: 1})
    
    # Salty Food Consumption: 1→5, 2→3, 3→1
    df['Salty Food Consumption'] = df['Salty Food Consumption'].replace({1: 5, 2: 3, 3: 1})
    
    # Sugar-Sweetened Beverages: 1→5, 2→3, 3 or 4→1
    df['Sugar-Sweetened Beverages'] = df['Sugar-Sweetened Beverages'].replace({1: 5, 2: 3, 3: 1, 4: 1})
    
    # Additional Salt Use: 1→5, 2→3, 3→1
    df['Additional Salt Use'] = df['Additional Salt Use'].replace({1: 5, 2: 3, 3: 1})
    
    return df

total_df_only = transform_values(total_df_only)

total_df_only.to_csv('processed_data/total_only.csv', index='R-ID')